# EMNIST FPGA Live Cloud Classification

This notebook documents the **live OV7670-camera-to-Cloud Run EMNIST classification proof of concept** on the ZedBoard Rev E. It follows the organization of the MNIST live cloud notebook while recording only the differences required for EMNIST Balanced (47 classes).

Unlike the separate **EMNIST classification measurement** project, this implementation captures and freezes a camera-derived 28 × 28 image, transfers it through shared PL/PS BRAM, requests remote classification, and presents the returned class on the ZedBoard LEDs. **Cloud Run model deployment is documented in a separate branch.**

## 1. Live cloud inference path

```text
OV7670 camera → ROI selection / thresholding
      ↓
frozen 28 × 28 image (784 bytes) in PL BRAM
      ↓
shared AXI BRAM mailbox (196 packed 32-bit words)
      ↓
Zynq ARM Processing System (lwIP / TCP / Mbed TLS)
      ↓
HTTPS POST /predict → Google Cloud Run EMNIST CNN
      ↓
JSON prediction (class ID 0–46)
      ↓
PS writes RESULT; PL reads prediction
      ↓
LED5–LED0 show the six-bit predicted class
```

The camera, preprocessing, mailbox and embedded networking workflow reused the MNIST implementation. The significant EMNIST changes were the remote **47-class CNN** and **six-bit prediction handling**.

## 2. Repository evidence

The following files are present at the **root of this branch**. The links are relative so they work after pushing the notebook to GitHub.

| File | Evidence |
|---|---|
| [emnist_cloud_live_classification.mp4](emnist_cloud_live_classification.mp4) | Video of successful live-camera classification |
| [emnist_cloud_overview_board.mp4](emnist_cloud_overview_board.mp4) | Video showing the physical board/setup |
| [emnist_live_blockdesign.png](emnist_live_blockdesign.png) | Vivado IP Integrator block design |
| [emnist_live_elaborted_rtl.png](emnist_live_elaborted_rtl.png) | Vivado elaborated RTL view (filename retained exactly as uploaded) |
| [EMNIST_FPGA_Cloud_Live_Classification_Development_and_Troubleshooting.md](EMNIST_FPGA_Cloud_Live_Classification_Development_and_Troubleshooting.md) | Development and troubleshooting record |

The repository also includes `README.md` and `LICENSE`. No UART log or still board photograph is required here: the two videos provide the demonstration evidence.

## 3. Vivado block design

The IP Integrator view documents the Zynq PS connection through AXI to the shared BRAM mailbox and its interface with the PL. This is the system-level schematic for the **cloud-assisted** design, not a local CNN schematic.

![EMNIST live cloud Vivado block design](emnist_live_blockdesign.png)

The `.bd` design source and implementation files belong in the corresponding live-classification **Code** branch.

## 4. Elaborated live RTL schematic

The elaborated RTL view provides supporting evidence of the live FPGA structure. The live deployment retains the OV7670 acquisition and control logic, VGA/frame buffer, processed-image BRAM, and mailbox controller. The controlled 100-image test ROM and local EMNIST CNN are not needed for cloud inference.

![EMNIST live cloud elaborated RTL](emnist_live_elaborted_rtl.png)

**Interpretation note:** An elaborated schematic documents project structure; resource totals should be taken from a report generated from the **same verified build**, not inferred from this image.

## 5. Physical demonstration and board overview

### Successful live classification

[Watch the EMNIST live classification demonstration](emnist_cloud_live_classification.mp4)

The researcher confirmed that a handwritten **F** was classified correctly after the live Vivado and SDK changes were built and tested. This establishes functional operation for that observed character, not a statistical camera-classification accuracy rate.

### Hardware overview

[Watch the EMNIST ZedBoard hardware overview](emnist_cloud_overview_board.mp4)

The videos are retained as demonstration evidence. They are not treated as controlled latency or throughput measurements.

In [ ]:
from pathlib import Path

expected_files = [
    "emnist_cloud_live_classification.mp4",
    "emnist_cloud_overview_board.mp4",
    "emnist_live_blockdesign.png",
    "emnist_live_elaborted_rtl.png",
    "EMNIST_FPGA_Cloud_Live_Classification_Development_and_Troubleshooting.md",
]

for filename in expected_files:
    path = Path(filename)
    print(f"{'FOUND  ' if path.is_file() else 'MISSING'} {filename}")

## 6. Capture and FPGA preprocessing

The existing OV7670 pipeline selected a 112 × 112 camera region using columns 24–135 and rows 4–115, sampled every fourth pixel in each direction, and stored a **28 × 28** input in BRAM. The implemented thresholding mapped the upper four capture bits greater than `0x7` to pixel value `0x7F`, and other samples to `0x00`.

The capture button triggered a freeze after the final ROI sample, keeping one complete image stable for cloud submission. The VGA path supported display of the live camera and stored thumbnail; these were reused from the MNIST proof of concept.

## 7. Live BRAM mailbox protocol

The ARM-addressable BRAM mailbox used base address `0x40000000` and the same primary offsets as MNIST:

| Offset | Content |
|---|---|
| `0x000–0x30C` | 196 × 32-bit image words containing 784 pixels |
| `0x310` | STATUS: `0` idle, `1` image ready, `2` result ready |
| `0x314` | Image index (live capture uses `0`) |
| `0x318` | Returned prediction (0–46) |
| `0x31C` | Expected label (`255` for live, unknown ground truth) |

The PL published the image and set `STATUS=1`. The PS read the image, sent the HTTPS request, wrote the classification to `RESULT`, and then set `STATUS=2`. The PL latched the prediction and cleared the mailbox before accepting another capture.

The expected-label sentinel `255` prevents treating an unlabeled physical camera image as a ground-truth error.

## 8. Embedded SDK client behavior

The Xilinx SDK application reused the MNIST DHCP, DNS, TCP and Mbed TLS connection flow. Once the ARM client obtained a camera image from the mailbox, it sent a binary request using:

```http
POST /predict HTTP/1.1
Content-Type: application/octet-stream
Content-Length: 784
Connection: keep-alive
```

The cloud response contained a JSON integer `prediction`, with valid EMNIST class IDs **0–46**. The ARM client wrote the result before asserting the result-ready mailbox status. The live code disabled automatic formal-result dumping and displayed live prediction information through UART during debugging.

The cloud service's model, deployment and server code are intentionally **not** duplicated in this notebook.

## 9. Returned prediction and board display

The live cloud top-module design used:

| Indicator | Meaning |
|---|---|
| LED7 | Result-ready indication |
| LED6 | Inference busy |
| LED5–LED0 | Six-bit EMNIST prediction (0–46) |

The capture button initiated inference using a frozen camera image, and the stored-image display switch retained its VGA role. The video documents the observed successful prediction of **F**; no numeric live-camera accuracy or diagnostic latency is claimed from this single example.

## 10. Adaptation and troubleshooting

The initial copy of the EMNIST project contained controlled measurement logic and the locally instantiated neural-network hierarchy. For the live cloud deployment, the project was changed to use stored camera pixels instead of the test ROM; the local `emnist_complete_cnn_bram_top` hierarchy and dataset-specific test ROM/labels were removed from the cloud-only path.

During SDK compilation, `main.c` produced cascading errors mentioning `ip_addr_t`, `struct netif`, `PLATFORM_EMAC_BASEADDR`, and an unexpected token. The researcher identified the actual cause as **a comment that had been split in two**. After correcting the malformed comment, the live application built and successfully predicted F.

See the full [Development and Troubleshooting Record](EMNIST_FPGA_Cloud_Live_Classification_Development_and_Troubleshooting.md) for the chronological changes and their limitations. The record was drafted before the final filenames were supplied, so its proposed `evidence/` and `schematics/` paths are superseded by the actual root-level filenames used here.

## 11. Security and evidence boundaries

The embedded development TLS configuration supported encrypted transport but **disabled server-certificate verification** and used **deterministic development entropy**, which was not cryptographically secure. A successful handshake did not establish authenticated server identity. These are implementation limitations, not production recommendations.

This branch demonstrates the working **live camera → FPGA → ARM → Cloud Run → FPGA** path. It does **not** contain the formal 1,000-image accuracy evaluation, 30 formal timing runs, continuous power-test measurements, or Cloud Run deployment code. Those belong to their respective measurement and cloud-service branches.

## 12. Related implementation branches and scope

- **EMNIST-FPGA-Cloud-Live-Classification-Code:** verified VHDL, Vivado block-design source and Xilinx SDK C application.
- **EMNIST-FPGA-Cloud-Classification-Measurement-Notebook-and-Data:** labeled-image runs, statistics, power observations and Vivado resource reports.
- **EMNIST-FPGA-Cloud-Classification-Measurement-Code:** controlled test ROM and measurement firmware.
- **Separate Cloud Run branch:** hosted EMNIST CNN and deployment-specific materials.

The principal evidence for **this notebook** is the two live demonstration videos, the two Vivado architecture views and the development troubleshooting record.